# Assignment 2: crime / violence classification

Decision the model supports: flag frames from city cameras that look like
`violence` or `weaponized` activity (labels as defined by the Roboflow dataset),
versus `normal`. Who acts, what each error costs, thresholds, and whether to ship
are graded decisions and stay in REPORT.md — this notebook produces the numbers
those decisions need.

**Dataset:** [`pdl-qtfdi/crime-detection-jekv6`](https://universe.roboflow.com/pdl-qtfdi/crime-detection-jekv6) version `1`
(image classification, three classes). We use the public export as labelled; we do
**not** fork it on Universe.

**Pipeline (built step by step):**
1. Download the Roboflow export and rebuild splits **by source video** (this step).
2. Inspect the data.
3. Fine-tune YOLO11 classification, compare baselines, cost the confusion matrix,
   save weights and artifacts for the Gradio app.

Run end-to-end with `uv run python main.py train` once the later sections exist.
Style follows the course card-detector notebook: short code, prose that says why.


## 0. Setup

Paths, device, and the same training hygiene as the card notebook: pin a seed for
shuffling, keep `deterministic=False` so GPU backends do not spam warnings, and
write runs under this repo rather than wherever Ultralytics last pointed.


In [ ]:
from __future__ import annotations

import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import torch

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab ships its own GPU PyTorch — do not uv sync here (see SETUP.md).
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "roboflow", "pillow"]
    )

ROOT = Path.cwd()
if not (ROOT / "api_key.py").is_file():
    raise SystemExit("Run this notebook from the assignment repo root.")

DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
SEED = 0

ROBOFLOW_WORKSPACE = "pdl-qtfdi"
ROBOFLOW_PROJECT = "crime-detection-jekv6"
ROBOFLOW_VERSION = 1

RAW_DIR = ROOT / "datasets" / "crime-detection-jekv6"
SPLIT_DIR = ROOT / "datasets" / "crime-detection-jekv6-videosplit"
CLASSES = ("normal", "violence", "weaponized")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIDEO_ID_RE = re.compile(r"^([nvw]\d+)_", re.IGNORECASE)

# Fractions of *videos per class* (confirmed): 70 / 15 / 15
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

WEIGHTS_DIR = ROOT / "weights"
ARTIFACTS_DIR = ROOT / "artifacts"

# Locked training defaults (Step 3)
MODEL_NAME = "yolo11n-cls.pt"
EPOCHS = 15
IMGSZ = 224
BATCH = 16

print(f"colab: {IN_COLAB}")
print(f"device: {DEVICE}")
print(f"raw export: {RAW_DIR}")
print(f"video-level split: {SPLIT_DIR}")
print(f"weights -> {WEIGHTS_DIR}")
print(f"train: {MODEL_NAME} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH}")


## 1. Download the public export

Key comes from `api_key.load_api_key()` (env var or `.env`). Format is Roboflow
**folder** classification: `train|valid|test/<class>/*.jpg`.

Roboflow's own split is image-level. Filenames are frames from source clips
(`v001_…_frame_40_…`). The audit showed most clips appear in more than one split,
so a model can memorise a scene in train and get credit for another frame of the
same scene in test. Section 2 rebuilds the folders so each video ID lands in
exactly one split.


In [ ]:
from api_key import load_api_key
from roboflow import Roboflow


def ensure_roboflow_export(destination: Path) -> Path:
    """Download version 1 once; reuse the folder on later runs."""
    marker = destination / "train"
    if marker.is_dir() and any(marker.iterdir()):
        print(f"already on disk: {destination}")
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    rf = Roboflow(api_key=load_api_key())
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    print(f"downloading {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION} …")
    project.version(ROBOFLOW_VERSION).download("folder", location=str(destination))
    return destination


ensure_roboflow_export(RAW_DIR)


## 2. Resplit by source video (70 / 15 / 15)

**Decision (confirmed):** assign videos, not images. Within each class, shuffle
video IDs with `seed=0`, then take 70% train / 15% valid / 15% test. All frames
(including Roboflow's 2× augment copies) of a video stay together.

Output: `datasets/crime-detection-jekv6-videosplit/`. Training later must read
**this** tree, not the raw Roboflow folders.


In [ ]:
def video_id_from_name(path: Path) -> str | None:
    match = VIDEO_ID_RE.match(path.name)
    return match.group(1).lower() if match else None


def collect_videos(raw_dir: Path) -> dict[str, dict]:
    """Map video_id -> class_name and file list."""
    videos: dict[str, dict] = {}
    for split in ("train", "valid", "test"):
        split_dir = raw_dir / split
        if not split_dir.is_dir():
            continue
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            class_name = class_dir.name
            for image in class_dir.rglob("*"):
                if image.suffix.lower() not in IMAGE_EXTS:
                    continue
                vid = video_id_from_name(image)
                if vid is None:
                    raise ValueError(f"cannot parse video id: {image.name}")
                entry = videos.setdefault(vid, {"class_name": class_name, "files": []})
                if entry["class_name"] != class_name:
                    raise ValueError(
                        f"video {vid} has conflicting classes "
                        f"{entry['class_name']} vs {class_name}"
                    )
                entry["files"].append(image)
    return videos


def allocate_counts(n: int) -> tuple[int, int, int]:
    """Integer counts that sum to n; leftovers go to train."""
    n_val = int(n * VAL_FRAC)
    n_test = int(n * TEST_FRAC)
    n_train = n - n_val - n_test
    return n_train, n_val, n_test


def assign_splits(videos: dict[str, dict], seed: int = SEED) -> dict[str, str]:
    """Return video_id -> train/valid/test, stratified by class."""
    by_class: dict[str, list[str]] = defaultdict(list)
    for vid, meta in videos.items():
        by_class[meta["class_name"]].append(vid)

    assignment: dict[str, str] = {}
    rng = random.Random(seed)
    for class_name in CLASSES:
        ids = sorted(by_class[class_name])
        rng.shuffle(ids)
        n_train, n_val, n_test = allocate_counts(len(ids))
        train_ids = ids[:n_train]
        val_ids = ids[n_train : n_train + n_val]
        test_ids = ids[n_train + n_val :]
        assert len(test_ids) == n_test
        for vid in train_ids:
            assignment[vid] = "train"
        for vid in val_ids:
            assignment[vid] = "valid"
        for vid in test_ids:
            assignment[vid] = "test"
        print(
            f"{class_name:12s} videos={len(ids):3d} -> "
            f"train {n_train}, valid {n_val}, test {n_test}"
        )
    return assignment


def write_videosplit(
    videos: dict[str, dict],
    assignment: dict[str, str],
    destination: Path,
) -> None:
    if destination.exists():
        shutil.rmtree(destination)
    for split in ("train", "valid", "test"):
        for class_name in CLASSES:
            (destination / split / class_name).mkdir(parents=True, exist_ok=True)

    for vid, meta in videos.items():
        split = assignment[vid]
        class_name = meta["class_name"]
        out_dir = destination / split / class_name
        for src in meta["files"]:
            shutil.copy2(src, out_dir / src.name)


def summarise_split(destination: Path) -> None:
    print("\nimage counts after video-level split:")
    grand = Counter()
    for split in ("train", "valid", "test"):
        counts = Counter()
        for class_name in CLASSES:
            folder = destination / split / class_name
            n = sum(1 for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS)
            counts[class_name] = n
            grand[class_name] += n
        total = sum(counts.values())
        parts = ", ".join(f"{c}={counts[c]}" for c in CLASSES)
        print(f"  {split:5s} n={total:5d}  ({parts})")
    print(f"  ALL   n={sum(grand.values()):5d}")


def assert_no_video_leakage(destination: Path) -> None:
    seen: dict[str, str] = {}
    leaks = []
    for split in ("train", "valid", "test"):
        for class_dir in (destination / split).iterdir():
            for image in class_dir.iterdir():
                if image.suffix.lower() not in IMAGE_EXTS:
                    continue
                vid = video_id_from_name(image)
                if vid in seen and seen[vid] != split:
                    leaks.append((vid, seen[vid], split))
                else:
                    seen[vid] = split
    if leaks:
        raise AssertionError(f"video leakage remains, e.g. {leaks[:5]}")
    print(f"\nleakage check: OK ({len(seen)} videos, each in exactly one split)")


videos = collect_videos(RAW_DIR)
print(f"source videos: {len(videos)}")
print(f"frames total:  {sum(len(v['files']) for v in videos.values())}")
print()
assignment = assign_splits(videos)
write_videosplit(videos, assignment, SPLIT_DIR)
summarise_split(SPLIT_DIR)
assert_no_video_leakage(SPLIT_DIR)
print(f"\nwrote {SPLIT_DIR}")


### Step 1 checkpoint

Leakage check above should read OK. The folders under `SPLIT_DIR` are what every
later cell uses. Next: look at composition and samples before training.


## 3. Look at the data

Never trust a split you have not looked at. Two checks matter here:

1. **Composition** — image counts vs video counts. Videos have different lengths, so
   a 70/15/15 video split will not land as 70/15/15 images. Say that in REPORT.md.
2. **Eyes on samples** — contact sheets per class. These frames come from short clips
   (`n###` / `v###` / `w###`), often with Roboflow's stretch-to-640 and 2× augment.
   Ask whether they look like the city CCTV story you framed, or like something else
   (acted scenes, close-ups, movie stills). That gap is a limitation either way; it
   does not by itself stop the assignment.


In [ ]:
from PIL import Image, ImageDraw, ImageFont


def split_composition(split_dir: Path = SPLIT_DIR) -> None:
    """Print image and unique-video counts per split and class."""
    print(f"composition for {split_dir.name}\n")
    header = f"{'split':6} {'class':12} {'images':>7} {'videos':>7} {'img%':>6}"
    print(header)
    print("-" * len(header))
    for split in ("train", "valid", "test"):
        split_total = 0
        rows = []
        for class_name in CLASSES:
            folder = split_dir / split / class_name
            files = [p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS]
            vids = {video_id_from_name(p) for p in files}
            rows.append((class_name, len(files), len(vids)))
            split_total += len(files)
        for class_name, n_img, n_vid in rows:
            share = 100 * n_img / split_total if split_total else 0
            print(f"{split:6} {class_name:12} {n_img:7d} {n_vid:7d} {share:5.1f}%")
        print(f"{split:6} {'TOTAL':12} {split_total:7d}")
        print()


def class_sheet(
    class_name: str,
    split: str = "train",
    count: int = 6,
    columns: int = 3,
    thumb: int = 224,
    seed: int = SEED,
):
    """Contact sheet of random frames for one class (reproducible seed)."""
    folder = SPLIT_DIR / split / class_name
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    rng = random.Random(seed)
    sample = files if len(files) <= count else rng.sample(files, count)
    rows = (len(sample) + columns - 1) // columns
    sheet = Image.new("RGB", (columns * thumb, rows * thumb), "black")
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    for position, path in enumerate(sample):
        image = Image.open(path).convert("RGB").resize((thumb, thumb))
        draw = ImageDraw.Draw(image)
        label = video_id_from_name(path) or path.stem[:12]
        draw.rectangle((0, 0, thumb, 14), fill=(0, 0, 0))
        draw.text((2, 1), f"{label}", fill=(0, 255, 0), font=font)
        sheet.paste(image, ((position % columns) * thumb, (position // columns) * thumb))
    return sheet


split_composition()
print("sample sheets (train, seed=0):")
for class_name in CLASSES:
    sheet = class_sheet(class_name)
    display_path = ROOT / "runs" / "inspect" / f"{class_name}_train_sheet.jpg"
    display_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(display_path)
    print(f"  saved {display_path.relative_to(ROOT)}")
    try:
        display(sheet)  # Jupyter
    except NameError:
        pass  # main.py train / plain script: files on disk are enough


In [ ]:
# When run via `main.py train` (nbconvert -> script), there is no IPython display().
# Re-show paths only; open the JPEGs under runs/inspect/ if you are not in a notebook UI.
inspect_dir = ROOT / "runs" / "inspect"
for class_name in CLASSES:
    path = inspect_dir / f"{class_name}_train_sheet.jpg"
    assert path.is_file(), path
    print(path.relative_to(ROOT), f"({path.stat().st_size} bytes)")


### What to take from the sheets

Write your own reading in REPORT.md. Prompts, not answers:

- Do `violence` and `weaponized` look separable from `normal` to a human at a glance?
- Are these closer to CCTV plazas or to staged / cinematic clips?
- Augment copies of the same frame will appear similar; that is expected from the
  Roboflow export, not a bug in our video split.

Image-count imbalance across splits is a consequence of uneven clip lengths. Class
balance inside each split should still be roughly even (we stratified by video class).


### Step 2 checkpoint

Composition and sheets are done. Next section fine-tunes on the video-level split.
If you train on Colab, run Sections 0–2 there first (same code), then Section 4.


## 4. Fine-tune YOLO11n-cls

Same pattern as the card notebook: start from a pretrained checkpoint, train on our
folder-format data, write runs under this repo, copy `best.pt` into `weights/` for
the app and for submission.

**Defaults (confirmed):** `yolo11n-cls.pt`, 15 epochs, `imgsz=224`, `batch=16`,
`seed=0`, `deterministic=False`. Data path: `SPLIT_DIR` (video-level split), not the
raw Roboflow folders.

### Training on Colab (recommended)

1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Get this repo onto Colab (upload the folder, or clone your fork). Working
   directory must contain `api_key.py` and this notebook.
3. Set the key: Colab secret `ROBOFLOW_API_KEY`, or a `.env` file, or
   `os.environ["ROBOFLOW_API_KEY"] = "..."` in a cell (do not commit that cell).
4. Run Sections 0 → 4 top to bottom. Skip nothing: the resplit must happen on Colab
   too unless you upload `datasets/crime-detection-jekv6-videosplit/` yourself.
5. **Download and commit back to the laptop repo:**
   - `weights/best.pt` (required)
   - optional: `runs/crime_videosplit/` plots if you want them for REPORT.md
6. Do **not** commit `datasets/`.

Local `uv run python main.py train` will re-run this pipeline on CPU (~1 hour) unless
you only need the committed weights for the app — graders use `check.py`, which does
not retrain by default.


In [ ]:
import json

from ultralytics import YOLO

assert SPLIT_DIR.is_dir() and any((SPLIT_DIR / "train").iterdir()), (
    f"Missing video-level split at {SPLIT_DIR}. Run Sections 1–2 first."
)

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"training on device={DEVICE} data={SPLIT_DIR}")
crime_model = YOLO(MODEL_NAME)

train_results = crime_model.train(
    data=str(SPLIT_DIR),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(ROOT / "runs"),
    name="crime_videosplit",
    exist_ok=True,
    seed=SEED,
    deterministic=False,
)

RUN_DIR = Path(crime_model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
DEST = WEIGHTS_DIR / "best.pt"
shutil.copy2(BEST_WEIGHTS, DEST)

print()
print("run directory:", RUN_DIR)
print("best weights: ", BEST_WEIGHTS, f"({BEST_WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print("copied to:    ", DEST)
print("classes:", YOLO(str(DEST)).names)

# Lightweight run record for REPORT.md / later app wiring (no cost decisions yet).
run_record = {
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "device": str(DEVICE),
    "data": str(SPLIT_DIR.relative_to(ROOT)) if SPLIT_DIR.is_relative_to(ROOT) else str(SPLIT_DIR),
    "roboflow": f"{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}",
    "split": "video-level 70/15/15 stratified by class",
    "weights": str(DEST.relative_to(ROOT)) if DEST.is_relative_to(ROOT) else str(DEST),
    "run_dir": str(RUN_DIR),
    "colab": bool(IN_COLAB),
}
(ARTIFACTS_DIR / "train_run.json").write_text(
    json.dumps(run_record, indent=2) + "\n", encoding="utf-8"
)
print("wrote", ARTIFACTS_DIR / "train_run.json")


In [ ]:
# Need json in this namespace when the train cell runs after a fresh kernel —
# import here so nbconvert order cannot bite us.
import json

results_png = RUN_DIR / "results.png"
if results_png.is_file():
    print("training curves:", results_png)
    try:
        from IPython.display import Image as IPyImage, display

        display(IPyImage(filename=str(results_png)))
    except Exception:
        pass
else:
    print("no results.png yet (unexpected for a finished Ultralytics run)")


### Step 3 checkpoint

When this finishes, you should have `weights/best.pt` and `artifacts/train_run.json`.

**If you are on Colab:** download those two files into the same paths in your local
clone, then commit `weights/best.pt` (and the json if you want).

**Next (Step 4, after you confirm):** evaluate on the **test** split vs majority-class
baseline and a zero-shot / pretrained baseline story; per-class metrics. Cost sweep
still waits on your euro costs, base rate, and positive-class definition.
